# 02 - Exploratory Data Analysis (EDA)

Comprehensive analysis of India's Air Quality dataset covering:
1. Missing value analysis
2. Pollutant distributions
3. Correlation analysis
4. Temporal trends (yearly, monthly, seasonal)
5. City-wise analysis
6. AQI category distribution & class imbalance
7. Outlier detection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from config import DATA_RAW, RAW_CSV, POLLUTANT_COLS, AQI_CATEGORIES, INDIA_SEASONS, FIGURES_DIR

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Load the original 2015-2020 CPCB dataset (26 cities, verified data)
df = pd.read_csv(RAW_CSV)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['City', 'Date']).reset_index(drop=True)

print(f"Dataset: Air Quality Data in India (2015-2020, CPCB)")
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Cities: {df['City'].nunique()} - {sorted(df['City'].unique())}")

---
## 1. Missing Value Analysis

In [ ]:
# Missing value percentages
pollutant_cols = [c for c in POLLUTANT_COLS if c in df.columns]
cols_of_interest = pollutant_cols + ['AQI', 'AQI_Bucket']

missing_pct = (df[cols_of_interest].isnull().sum() / len(df) * 100).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart of missing percentages
colors = ['#d32f2f' if x > 50 else '#f57c00' if x > 30 else '#388e3c' for x in missing_pct.values]
missing_pct.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_xlabel('Missing %')
axes[0].set_title('Missing Values by Column')
axes[0].axvline(x=30, color='orange', linestyle='--', alpha=0.7, label='30% threshold')
axes[0].legend()

# Missing value matrix (missingno)
msno.matrix(df[cols_of_interest], ax=axes[1], sparkline=False, fontsize=10)
axes[1].set_title('Missing Value Patterns (white = missing)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMissing value percentages:")
print(missing_pct.round(1).to_string())

In [ ]:
# Missing values by city
city_missing = df.groupby('City')[pollutant_cols].apply(lambda x: x.isnull().mean() * 100).round(1)

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(city_missing, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Missing %'})
ax.set_title('Missing Value % by City and Pollutant')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_missing_by_city.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Pollutant Distributions

In [ ]:
# Histograms + boxplots for each pollutant
key_pollutants = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3']
key_pollutants = [p for p in key_pollutants if p in df.columns]

fig, axes = plt.subplots(len(key_pollutants), 2, figsize=(16, 4 * len(key_pollutants)))

for i, col in enumerate(key_pollutants):
    data = df[col].dropna()
    
    # Histogram
    axes[i, 0].hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i, 0].axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.1f}')
    axes[i, 0].axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.1f}')
    axes[i, 0].set_title(f'{col} Distribution')
    axes[i, 0].legend(fontsize=9)
    
    # Boxplot
    axes[i, 1].boxplot(data, vert=False, widths=0.7,
                        boxprops=dict(color='steelblue'),
                        medianprops=dict(color='red'))
    axes[i, 1].set_title(f'{col} Box Plot')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_pollutant_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# AQI Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

aqi_data = df['AQI'].dropna()
axes[0].hist(aqi_data, bins=60, color='coral', edgecolor='white', alpha=0.8)
axes[0].axvline(aqi_data.mean(), color='darkred', linestyle='--', label=f'Mean: {aqi_data.mean():.0f}')
axes[0].set_title('AQI Distribution')
axes[0].set_xlabel('AQI')
axes[0].legend()

# AQI by category with color coding
cat_colors = {'Good': '#00e400', 'Satisfactory': '#92d050', 'Moderate': '#ffff00',
              'Poor': '#ff7e00', 'Very Poor': '#ff0000', 'Severe': '#99004c'}
bucket_counts = df['AQI_Bucket'].value_counts()
bucket_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
bucket_counts = bucket_counts.reindex([b for b in bucket_order if b in bucket_counts.index])
bars = axes[1].bar(bucket_counts.index, bucket_counts.values,
                    color=[cat_colors.get(b, 'gray') for b in bucket_counts.index],
                    edgecolor='black', alpha=0.85)
axes[1].set_title('AQI Category Distribution')
axes[1].set_ylabel('Count')
for bar, val in zip(bars, bucket_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{val}\n({val/bucket_counts.sum()*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_aqi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Correlation Analysis

In [ ]:
# Correlation heatmap
corr_cols = [c for c in key_pollutants + ['AQI'] if c in df.columns]
corr_matrix = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pearson
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Pearson Correlation')

# Spearman
spearman_corr = df[corr_cols].corr(method='spearman')
sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Spearman Correlation')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

# Print correlation with AQI
print("\nCorrelation with AQI (Pearson):")
print(corr_matrix['AQI'].drop('AQI').sort_values(ascending=False).round(3).to_string())

---
## 4. Temporal Trends

In [ ]:
# Add temporal columns
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day_of_week'] = df['Date'].dt.dayofweek
df['season'] = df['month'].map(INDIA_SEASONS)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Yearly trend
yearly = df.groupby('year')['AQI'].agg(['mean', 'median']).dropna()
axes[0, 0].plot(yearly.index, yearly['mean'], 'o-', color='coral', label='Mean AQI', linewidth=2)
axes[0, 0].plot(yearly.index, yearly['median'], 's--', color='steelblue', label='Median AQI', linewidth=2)
axes[0, 0].set_title('Yearly AQI Trend')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('AQI')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Monthly pattern
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly = df.groupby('month')['AQI'].agg(['mean', 'std'])
axes[0, 1].bar(range(1, 13), monthly['mean'], yerr=monthly['std'], capsize=3,
               color=['#d32f2f' if m in [11, 12, 1] else '#388e3c' if m in [7, 8, 9] else '#f57c00' for m in range(1, 13)],
               alpha=0.8, edgecolor='black')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].set_xticklabels(month_names)
axes[0, 1].set_title('Monthly AQI Pattern')
axes[0, 1].set_ylabel('Mean AQI')

# Seasonal pattern
season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
season_colors = {'Winter': '#1565c0', 'Summer': '#ff8f00', 'Monsoon': '#2e7d32', 'Post-Monsoon': '#6a1b9a'}
seasonal = df.groupby('season')['AQI'].describe()[['mean', '50%', 'std']]
seasonal = seasonal.reindex(season_order)
bars = axes[1, 0].bar(seasonal.index, seasonal['mean'],
                        color=[season_colors[s] for s in seasonal.index],
                        edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Seasonal AQI (India-specific seasons)')
axes[1, 0].set_ylabel('Mean AQI')
for bar, val in zip(bars, seasonal['mean']):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                     f'{val:.0f}', ha='center', va='bottom', fontweight='bold')

# Day of week pattern
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow = df.groupby('day_of_week')['AQI'].mean()
axes[1, 1].bar(range(7), dow.values, color='steelblue', edgecolor='black', alpha=0.8)
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(dow_names)
axes[1, 1].set_title('Day of Week AQI Pattern')
axes[1, 1].set_ylabel('Mean AQI')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_temporal_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Time series for top 5 most polluted cities
top5_cities = df.groupby('City')['AQI'].mean().nlargest(5).index.tolist()

fig, ax = plt.subplots(figsize=(18, 6))
for city in top5_cities:
    city_data = df[df['City'] == city].set_index('Date')['AQI'].resample('ME').mean()
    ax.plot(city_data.index, city_data.values, label=city, linewidth=1.5, alpha=0.8)

ax.set_title('Monthly Average AQI - Top 5 Most Polluted Cities')
ax.set_ylabel('AQI')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_top5_cities_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. City-wise Analysis

In [ ]:
# City-wise mean AQI (sorted)
city_aqi = df.groupby('City')['AQI'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['#d32f2f' if v > 200 else '#ff9800' if v > 150 else '#4caf50' for v in city_aqi.values]
city_aqi.plot(kind='barh', ax=ax, color=colors, edgecolor='black', alpha=0.85)
ax.set_xlabel('Mean AQI')
ax.set_title('City-wise Average AQI (2015-2020)')
ax.axvline(x=100, color='green', linestyle='--', alpha=0.5, label='Satisfactory threshold')
ax.axvline(x=200, color='red', linestyle='--', alpha=0.5, label='Poor threshold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_city_mean_aqi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# City x Month heatmap
city_month_aqi = df.pivot_table(values='AQI', index='City', columns='month', aggfunc='mean')
city_month_aqi.columns = month_names

# Sort by overall mean AQI
city_month_aqi = city_month_aqi.loc[city_month_aqi.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(city_month_aqi, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Mean AQI'})
ax.set_title('Mean AQI: City x Month Heatmap')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '09_city_month_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. AQI Category Distribution & Class Imbalance

In [ ]:
# Class distribution analysis
bucket_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
cat_colors_list = ['#00e400', '#92d050', '#ffff00', '#ff7e00', '#ff0000', '#99004c']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Overall distribution
counts = df['AQI_Bucket'].value_counts().reindex([b for b in bucket_order if b in df['AQI_Bucket'].unique()])
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[cat_colors_list[bucket_order.index(b)] for b in counts.index],
            startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Overall AQI Category Distribution')

# By season
season_bucket = pd.crosstab(df['season'], df['AQI_Bucket'], normalize='index') * 100
season_bucket = season_bucket.reindex(columns=[b for b in bucket_order if b in season_bucket.columns])
season_bucket = season_bucket.reindex(season_order)
season_bucket.plot(kind='bar', stacked=True, ax=axes[1],
                    color=[cat_colors_list[bucket_order.index(b)] for b in season_bucket.columns],
                    edgecolor='black', alpha=0.85)
axes[1].set_title('AQI Category % by Season')
axes[1].set_ylabel('Percentage')
axes[1].legend(title='AQI Bucket', bbox_to_anchor=(1.0, 1), fontsize=8)
axes[1].tick_params(axis='x', rotation=30)

# Imbalance ratio
imbalance = counts / counts.min()
axes[2].bar(imbalance.index, imbalance.values, color='steelblue', edgecolor='black', alpha=0.8)
axes[2].set_title('Class Imbalance Ratio (relative to smallest class)')
axes[2].set_ylabel('Ratio')
axes[2].tick_params(axis='x', rotation=30)
for i, (idx, val) in enumerate(zip(imbalance.index, imbalance.values)):
    axes[2].text(i, val + 0.3, f'{val:.1f}x', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClass counts:")
print(counts.to_string())

---
## 7. Outlier Detection

In [ ]:
# IQR-based outlier analysis
outlier_summary = []
for col in key_pollutants:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = max(Q1 - 1.5 * IQR, 0), Q3 + 1.5 * IQR
    outliers = ((data < lower) | (data > upper)).sum()
    outlier_summary.append({
        'Pollutant': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
        'IQR': round(IQR, 2), 'Lower': round(lower, 2), 'Upper': round(upper, 2),
        'Outliers': outliers, 'Outlier_Pct': round(outliers / len(data) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

In [ ]:
# Scatter plots: each key pollutant vs AQI with outliers highlighted
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(key_pollutants):
    data = df[[col, 'AQI']].dropna()
    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    
    is_outlier = data[col] > upper
    axes[i].scatter(data.loc[~is_outlier, col], data.loc[~is_outlier, 'AQI'],
                     alpha=0.2, s=5, color='steelblue', label='Normal')
    axes[i].scatter(data.loc[is_outlier, col], data.loc[is_outlier, 'AQI'],
                     alpha=0.5, s=8, color='red', label='Outlier')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('AQI')
    axes[i].set_title(f'{col} vs AQI')
    axes[i].legend(fontsize=8)

# Hide unused subplot
if len(key_pollutants) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_outlier_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Key EDA Findings Summary

### Missing Values
- Xylene, Toluene, Benzene have extremely high missing rates (>60%)
- PM2.5, PM10, NO2 have moderate missing rates (~20-30%)
- Multi-tier imputation strategy needed

### Distributions
- Most pollutants are right-skewed (long tail of high-pollution events)
- PM2.5 and PM10 show the widest range and strongest AQI correlation

### Temporal Patterns
- **Winter** (Nov-Feb) has the worst AQI - cold air traps pollutants (temperature inversion)
- **Monsoon** (Jun-Sep) has the cleanest air - rain washes out particulates
- **Post-Monsoon** (Oct-Nov) shows sharp AQI spike - crop burning in Punjab/Haryana
- Day-of-week effect is minimal

### City Patterns
- North Indian cities (Delhi, Patna, Lucknow) consistently most polluted
- Coastal cities (Thiruvananthapuram, Shillong) have cleanest air

### Class Imbalance
- "Good" and "Moderate" dominate the dataset
- "Severe" category is rare (~1-3%) - will need SMOTE or class weights

### Outliers
- CO and NH3 have the highest outlier percentages
- Winsorization (capping) recommended to preserve data

### Next: Notebook 03 - Preprocessing & Feature Engineering